# Kaggle TF-IDF Logistic Regression A/B Swap Submission

This notebook trains TF-IDF + Logistic Regression with A/B swap augmentation on the full training set and saves `/kaggle/working/submission.csv`.

## 1. Imports and Auto Path Search

Find Kaggle input files automatically under `/kaggle/input`.

In [ ]:
import ast
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
LABELS = [0, 1, 2]

INPUT_ROOT = Path('/kaggle/input')
submission_path = Path('/kaggle/working/submission.csv')


def find_input_file(filename):
    paths = sorted(INPUT_ROOT.rglob(filename))
    print(f'Found {filename} paths:')
    for path in paths:
        print(path)
    if not paths:
        raise FileNotFoundError(f'Could not find {filename} under {INPUT_ROOT}')
    return paths[0]


train_path = find_input_file('train.csv')
test_path = find_input_file('test.csv')
sample_submission_path = find_input_file('sample_submission.csv')

print(f'Input root: {INPUT_ROOT}')
print(f'train.csv exists: {train_path.exists()} -> {train_path}')
print(f'test.csv exists: {test_path.exists()} -> {test_path}')
print(f'sample_submission.csv exists: {sample_submission_path.exists()} -> {sample_submission_path}')
print(f'Output submission path: {submission_path}')

## 2. Read Data

Read official Kaggle train, test, and sample submission files.

In [ ]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print(f'train shape: {train.shape}')
print(f'test shape: {test.shape}')
print(f'sample_submission shape: {sample_submission.shape}')

print('\ntrain columns:')
print(train.columns.tolist())

print('\ntest columns:')
print(test.columns.tolist())

print('\nsample_submission columns:')
print(sample_submission.columns.tolist())

## 3. Cleaning and Feature Functions

Parse JSON/list-like text fields, clean invalid Unicode, build model input text, and create A/B swapped data.

In [ ]:
def clean_unicode(text):
    if text is None:
        return ''
    if not isinstance(text, str):
        text = str(text)

    try:
        text = text.encode('utf-16', 'surrogatepass').decode('utf-16', 'replace')
    except Exception:
        text = text.encode('utf-8', 'ignore').decode('utf-8', 'ignore')

    text = ''.join(ch for ch in text if not (0xD800 <= ord(ch) <= 0xDFFF))
    return text


def parse_text_field(x):
    if not isinstance(x, str):
        x = str(x)

    text = x.strip()

    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        pass

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', SyntaxWarning)
            return ast.literal_eval(text)
    except (ValueError, SyntaxError, TypeError):
        return text


def normalize_text(x):
    if x is None:
        return ''

    try:
        if pd.isna(x):
            return ''
    except (TypeError, ValueError):
        pass

    if not isinstance(x, str):
        x = str(x)

    parsed = parse_text_field(x.strip())

    if isinstance(parsed, list):
        text = '\n'.join(str(item) for item in parsed if item is not None)
    else:
        text = str(parsed)

    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace(chr(0x2028), '\n').replace(chr(0x2029), '\n').strip()
    text = clean_unicode(text)
    return text


def make_text_input(df):
    prompt = df['prompt_clean'].fillna('').astype(str)
    response_a = df['response_a_clean'].fillna('').astype(str)
    response_b = df['response_b_clean'].fillna('').astype(str)

    return (
        'Prompt:\n' + prompt
        + '\n\nResponse A:\n' + response_a
        + '\n\nResponse B:\n' + response_b
    )


def swap_ab_dataframe(df):
    swapped = df.copy()
    swapped['response_a_clean'] = df['response_b_clean'].values
    swapped['response_b_clean'] = df['response_a_clean'].values

    if 'label' in swapped.columns:
        swapped['label'] = df['label'].map({0: 1, 1: 0, 2: 2}).astype(int).values

    return swapped


print('Functions ready.')

## 4. Clean Text and Create Labels

Create clean text fields and convert one-hot winner columns into labels.

In [ ]:
required_train_columns = {
    'id',
    'prompt',
    'response_a',
    'response_b',
    'winner_model_a',
    'winner_model_b',
    'winner_tie',
}
required_test_columns = {'id', 'prompt', 'response_a', 'response_b'}

missing_train_columns = sorted(required_train_columns - set(train.columns))
missing_test_columns = sorted(required_test_columns - set(test.columns))

if missing_train_columns:
    raise ValueError(f'train.csv missing columns: {missing_train_columns}')
if missing_test_columns:
    raise ValueError(f'test.csv missing columns: {missing_test_columns}')

text_columns = ['prompt', 'response_a', 'response_b']

for df_name, df in [('train', train), ('test', test)]:
    for column in text_columns:
        df[f'{column}_clean'] = df[column].apply(normalize_text)
    print(f'{df_name}: clean text columns created.')

train['label'] = np.select(
    [
        train['winner_model_a'] == 1,
        train['winner_model_b'] == 1,
        train['winner_tie'] == 1,
    ],
    [0, 1, 2],
    default=-1,
).astype(int)

if (train['label'] == -1).any():
    invalid_count = int((train['label'] == -1).sum())
    raise ValueError(f'Invalid label rows: {invalid_count}')

print('\nLabel counts:')
print(train['label'].value_counts().sort_index())

print('\nClean text example:')
print(train.loc[0, 'prompt_clean'][:500])

## 5. A/B Swap Augmentation on Full Training Data

Keep original train rows, add swapped copies, and train on the combined augmented dataset.

In [ ]:
train_original = train.copy()
swapped_train = swap_ab_dataframe(train_original)
train_aug = pd.concat([train_original, swapped_train], ignore_index=True)
train_aug = train_aug.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

train_aug['text_input'] = make_text_input(train_aug)
test['text_input'] = make_text_input(test)

print(f'Original train size: {len(train_original)}')
print(f'Swapped train size: {len(swapped_train)}')
print(f'Augmented train size: {len(train_aug)}')

print('\nOriginal label counts:')
print(train_original['label'].value_counts().sort_index())

print('\nSwapped label counts:')
print(swapped_train['label'].value_counts().sort_index())

print('\nAugmented label counts:')
print(train_aug['label'].value_counts().sort_index())

## 6. Train TF-IDF Logistic Regression

Fit the tuned TF-IDF Logistic Regression model on the full augmented training set.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32,
)

X_train = vectorizer.fit_transform(train_aug['text_input'])
y_train = train_aug['label'].astype(int)

print(f'X_train shape: {X_train.shape}')

model = LogisticRegression(
    C=0.1,
    max_iter=1000,
    solver='saga',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

model.fit(X_train, y_train)

print('Model training finished.')
print(f'Classes: {model.classes_.tolist()}')

## 7. Predict Original and Swapped Test

Predict original test order and swapped test order, map swapped probabilities back, then average.

In [ ]:
def align_probabilities(probabilities, classes, labels=LABELS):
    aligned = np.zeros((probabilities.shape[0], len(labels)), dtype=float)
    class_to_position = {int(label): idx for idx, label in enumerate(classes)}

    for output_position, label in enumerate(labels):
        if label in class_to_position:
            aligned[:, output_position] = probabilities[:, class_to_position[label]]

    return aligned


X_test_original = vectorizer.transform(test['text_input'])
probs_original_raw = model.predict_proba(X_test_original)
probs_original = align_probabilities(probs_original_raw, model.classes_, LABELS)

test_swapped = swap_ab_dataframe(test.copy())
test_swapped['text_input'] = make_text_input(test_swapped)
X_test_swapped = vectorizer.transform(test_swapped['text_input'])
probs_swapped_raw = model.predict_proba(X_test_swapped)
probs_swapped = align_probabilities(probs_swapped_raw, model.classes_, LABELS)

mapped_probs_swapped = np.column_stack([
    probs_swapped[:, 1],
    probs_swapped[:, 0],
    probs_swapped[:, 2],
])

final_probs = 0.5 * probs_original + 0.5 * mapped_probs_swapped

print(f'probs_original shape: {probs_original.shape}')
print(f'probs_swapped shape: {probs_swapped.shape}')
print(f'final_probs shape: {final_probs.shape}')

## 8. Create and Check Submission

Create Kaggle submission and validate columns, probability sums, and missing values.

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

required_submission_columns = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
probability_columns = ['winner_model_a', 'winner_model_b', 'winner_tie']
probability_sum = submission[probability_columns].sum(axis=1)

print(f'submission shape: {submission.shape}')
print(f'submission columns: {submission.columns.tolist()}')
print(f'submission columns correct: {submission.columns.tolist() == required_submission_columns}')

print('\nProbability sum describe:')
print(probability_sum.describe())
print(f'All probability sums close to 1: {np.allclose(probability_sum, 1.0, atol=1e-6)}')

print('\nNaN check:')
print(submission.isna().sum())

if submission.columns.tolist() != required_submission_columns:
    raise ValueError('Submission columns are not correct.')
if not np.allclose(probability_sum, 1.0, atol=1e-6):
    raise ValueError('Submission probability rows do not sum to 1.')
if submission.isna().any().any():
    raise ValueError('Submission contains NaN values.')

display(submission.head())
print('Submission checks passed.')

## 9. Save Submission

Save final output to `/kaggle/working/submission.csv`.

In [ ]:
submission.to_csv(submission_path, index=False)

print(f'Saved submission: {submission_path}')
print('Kaggle TF-IDF A/B swap submission finished successfully.')